# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")
df.display()

# Silver Transformation

Dynamically trims all string columns in a DataFrame, rather than manually specifying each column name.

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Clean Dates

The three date columns are stored as integers instead of proper date types. The integers are in YYYYMMDD format:

- 20101229 = Year 2010, Month 12, Day 29
- 20110105 = Year 2011, Month 01, Day 05

In [0]:
# Check for invalid dates across all three date columns
from pyspark.sql.functions import length, lit

date_cols = ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]

for col_name in date_cols:
    invalid_count = df.filter(
        (col(col_name) == 0) | (length(col(col_name).cast("string")) != 8)
    ).count()
    
    print(f"{col_name}: {invalid_count} invalid dates found")

# Show sample of invalid dates
print("\nSample of rows with invalid dates:")
df.filter(
    (col("sls_order_dt") == 0) | (length(col("sls_order_dt").cast("string")) != 8) |
    (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt").cast("string")) != 8) |
    (col("sls_due_dt") == 0) | (length(col("sls_due_dt").cast("string")) != 8)
).limit(10).display()

In [0]:
# Defensive date conversion - handles zeros and invalid formats
from pyspark.sql.functions import length, when, to_date

df = (
    df
    .withColumn(
        "sls_order_dt",
        when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt").cast("string")) != 8),
            None
        ).otherwise(to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt").cast("string")) != 8),
            None
        ).otherwise(to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt").cast("string")) != 8),
            None
        ).otherwise(to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

# Verify conversion
print("Date columns after conversion:")
df.select("sls_order_dt", "sls_ship_dt", "sls_due_dt").limit(5).display()

In [0]:
df.display()

# Sales and Price Correction

In [0]:
df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

# Rename Columns

In [0]:

RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

In [0]:
%sql
select * from workspace.silver.crm_sales limit 5